# Fase 4 - Componente Práctico: Sistema Integral de Gestión de Clientes, Servicios y Reservas (Software FJ)
**Curso:** Programación (213023) – UNAD  
**Estudiante:** Brayan Stiven García Calderón  
**Modalidad:** Desarrollo individual (realizado de forma individual debido a la falta de participación oportuna de los demás integrantes del grupo en el foro colaborativo).  

---
### Descripción de la Solución
Este cuaderno implementa una aplicación orientada a objetos en Python sin uso de bases de datos para **Software FJ**, integrando:
- **Abstracción y Herencia:** Clases abstractas `EntidadSistema` y `Servicio`, con clases derivadas `ReservaSala`, `AlquilerEquipo` y `AsesoriaEspecializada`.
- **Encapsulación y Validaciones:** Clase `Cliente` con propiedades (`@property`), atributos privados y validación estricta de datos.
- **Polimorfismo y Sobrecarga:** Métodos sobrescritos (`calcular_costo`, `describir_servicio`, `validar_parametros`) y método sobrecargado mediante parámetros opcionales (`calcular_costo_detallado` con impuestos, descuentos y recargos).
- **Manejo Avanzado de Excepciones:** Jerarquía de excepciones personalizadas, bloques `try/except`, `try/except/else`, `try/except/finally`, encadenamiento de excepciones (`raise ... from ...`) y registro persistente en `software_fj.log`.

In [1]:
import os  # Importa el módulo os para gestión de rutas de archivos
import re  # Importa el módulo re para validación con expresiones regulares
import logging  # Importa logging para registrar eventos y errores en archivo
from abc import ABC, abstractmethod  # Importa ABC y abstractmethod para definir clases abstractas
from datetime import datetime  # Importa datetime para marcas de tiempo y fechas de reserva
from typing import List, Optional  # Importa anotaciones de tipo para mayor claridad

# ==============================================================================
# 1. CONFIGURACIÓN DEL SISTEMA DE LOGS (ARCHIVO PLANO)
# ==============================================================================
LOG_FILENAME = "software_fj.log"  # Define el nombre del archivo de registro de eventos y errores

logger = logging.getLogger("SoftwareFJ")  # Crea una instancia de logger con nombre específico
logger.setLevel(logging.DEBUG)  # Establece el nivel mínimo de registro en DEBUG
if logger.hasHandlers():  # Verifica si el logger ya tiene manejadores activos para evitar duplicados
    logger.handlers.clear()  # Limpia los manejadores previos en ejecuciones repetidas del notebook

file_handler = logging.FileHandler(LOG_FILENAME, mode="w", encoding="utf-8")  # Crea manejador de archivo en modo escritura limpia
formatter = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")  # Define formato de línea del log
file_handler.setFormatter(formatter)  # Asigna el formato al manejador de archivo
logger.addHandler(file_handler)  # Agrega el manejador de archivo al logger


# ==============================================================================
# 2. JERARQUÍA DE EXCEPCIONES PERSONALIZADAS
# ==============================================================================
class ErrorSoftwareFJ(Exception):
    """Excepción base para todos los errores controlados del sistema Software FJ."""
    def __init__(self, mensaje: str, codigo_error: str = "ERR-000"):
        super().__init__(mensaje)  # Inicializa la clase padre Exception con el mensaje descriptivo
        self.codigo_error = codigo_error  # Almacena un código identificador del error


class DatosInvalidosError(ErrorSoftwareFJ):
    """Se lanza cuando los datos ingresados por el usuario o sistema no cumplen el formato."""
    def __init__(self, mensaje: str):
        super().__init__(mensaje, codigo_error="ERR-101")  # Asigna código específico de validación


class ParametroFaltanteError(ErrorSoftwareFJ):
    """Se lanza cuando falta un parámetro obligatorio o es nulo/vacío."""
    def __init__(self, mensaje: str):
        super().__init__(mensaje, codigo_error="ERR-102")  # Asigna código de parámetro faltante


class ServicioNoDisponibleError(ErrorSoftwareFJ):
    """Se lanza cuando se intenta reservar o usar un servicio inactivo o sin cupo."""
    def __init__(self, mensaje: str):
        super().__init__(mensaje, codigo_error="ERR-201")  # Asigna código de disponibilidad


class ReservaInvalidaError(ErrorSoftwareFJ):
    """Se lanza cuando una operación sobre una reserva viola las reglas de negocio."""
    def __init__(self, mensaje: str):
        super().__init__(mensaje, codigo_error="ERR-202")  # Asigna código de error en reserva


class CalculoInconsistenteError(ErrorSoftwareFJ):
    """Se lanza cuando un cálculo financiero arroja valores negativos o inconsistentes."""
    def __init__(self, mensaje: str):
        super().__init__(mensaje, codigo_error="ERR-301")  # Asigna código de error matemático/financiero


print("Módulo de logs y excepciones personalizadas inicializado correctamente.")

Módulo de logs y excepciones personalizadas inicializado correctamente.

In [2]:
# ==============================================================================
# 3. CLASE ABSTRACTA BASE DEL SISTEMA
# ==============================================================================
class EntidadSistema(ABC):
    """Clase abstracta que representa cualquier entidad general dentro de Software FJ."""

    def __init__(self, identificador: str):
        if not identificador or not str(identificador).strip():  # Valida que el ID no esté vacío
            raise ParametroFaltanteError("El identificador de la entidad no puede estar vacío.")  # Lanza excepción si falta ID
        self._id = str(identificador).strip()  # Encapsula el identificador como atributo protegido
        self._fecha_creacion = datetime.now()  # Registra la fecha y hora exacta de creación

    @property
    def id(self) -> str:
        """Retorna el identificador único de la entidad."""
        return self._id  # Devuelve el ID encapsulado

    @property
    def fecha_creacion(self) -> datetime:
        """Retorna la marca de tiempo de creación de la entidad."""
        return self._fecha_creacion  # Devuelve la fecha de creación

    @abstractmethod
    def obtener_resumen(self) -> str:
        """Método abstracto obligatorio para describir la entidad."""
        pass  # Debe ser implementado por cada clase hija


# ==============================================================================
# 4. CLASE CLIENTE CON ENCAPSULACIÓN Y VALIDACIONES ESTRICTAS
# ==============================================================================
class Cliente(EntidadSistema):
    """Representa un cliente de Software FJ con validación estricta de datos personales."""

    def __init__(self, id_cliente: str, nombre: str, correo: str, telefono: str):
        super().__init__(id_cliente)  # Invoca al constructor de la clase abstracta EntidadSistema
        self.nombre = nombre  # Usa el setter de nombre para ejecutar validación
        self.correo = correo  # Usa el setter de correo para ejecutar validación
        self.telefono = telefono  # Usa el setter de teléfono para ejecutar validación
        self.__historial_reservas: List[str] = []  # Lista privada para almacenar IDs de reservas del cliente
        logger.info(f"Cliente registrado exitosamente: [{self.id}] {self.nombre}")  # Registra evento en log

    @property
    def nombre(self) -> str:
        """Obtiene el nombre completo del cliente."""
        return self.__nombre  # Retorna atributo privado __nombre

    @nombre.setter
    def nombre(self, valor: str):
        """Valida y asigna el nombre del cliente."""
        if valor is None or not str(valor).strip():  # Verifica que el nombre no sea nulo ni vacío
            raise ParametroFaltanteError("El nombre del cliente es obligatorio.")  # Lanza excepción si falta
        nombre_limpio = str(valor).strip()  # Elimina espacios sobrantes
        if len(nombre_limpio) < 3 or any(char.isdigit() for char in nombre_limpio):  # Valida longitud mínima y ausencia de números
            raise DatosInvalidosError(f"Nombre de cliente inválido: '{valor}'. Debe tener al menos 3 letras y sin números.")
        self.__nombre = nombre_limpio  # Almacena el nombre validado en atributo privado

    @property
    def correo(self) -> str:
        """Obtiene el correo electrónico del cliente."""
        return self.__correo  # Retorna atributo privado __correo

    @correo.setter
    def correo(self, valor: str):
        """Valida el formato del correo electrónico mediante expresión regular."""
        if valor is None or not str(valor).strip():  # Verifica si el correo está vacío
            raise ParametroFaltanteError("El correo electrónico del cliente es obligatorio.")
        patron = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"  # Expresión regular estándar para email
        if not re.match(patron, str(valor).strip()):  # Comprueba coincidencia con el patrón
            raise DatosInvalidosError(f"Formato de correo electrónico inválido: '{valor}'.")
        self.__correo = str(valor).strip().lower()  # Guarda el correo normalizado en minúsculas

    @property
    def telefono(self) -> str:
        """Obtiene el número de teléfono del cliente."""
        return self.__telefono  # Retorna atributo privado __telefono

    @telefono.setter
    def telefono(self, valor: str):
        """Valida que el teléfono contenga únicamente dígitos y longitud válida."""
        if valor is None or not str(valor).strip():  # Verifica que no esté vacío
            raise ParametroFaltanteError("El número de teléfono del cliente es obligatorio.")
        digitos = re.sub(r"[\s\-\+]", "", str(valor))  # Limpia espacios, guiones o signo más
        if not digitos.isdigit() or not (7 <= len(digitos) <= 15):  # Exige entre 7 y 15 dígitos numéricos
            raise DatosInvalidosError(f"Teléfono inválido: '{valor}'. Debe contener entre 7 y 15 dígitos.")
        self.__telefono = digitos  # Asigna el número validado

    def agregar_reserva_historial(self, id_reserva: str):
        """Registra el ID de una reserva en el historial privado del cliente."""
        self.__historial_reservas.append(id_reserva)  # Añade el identificador a la lista interna

    def obtener_historial(self) -> List[str]:
        """Devuelve una copia segura del historial de reservas del cliente."""
        return list(self.__historial_reservas)  # Retorna copia para proteger la lista interna

    def obtener_resumen(self) -> str:
        """Implementa el método abstracto de EntidadSistema para describir al cliente."""
        return f"Cliente [{self.id}]: {self.nombre} | Email: {self.correo} | Tel: {self.telefono}"  # Formatea resumen


print("Clase abstracta EntidadSistema y clase Cliente cargadas correctamente.")

Clase abstracta EntidadSistema y clase Cliente cargadas correctamente.

In [3]:
# ==============================================================================
# 5. CLASE ABSTRACTA SERVICIO Y SERVICIOS ESPECIALIZADOS (HERENCIA Y POLIMORFISMO)
# ==============================================================================
class Servicio(EntidadSistema, ABC):
    """Clase abstracta que define la estructura base de los servicios de Software FJ."""

    def __init__(self, id_servicio: str, nombre_servicio: str, tarifa_base: float, disponible: bool = True):
        super().__init__(id_servicio)  # Llama al constructor de EntidadSistema
        if not nombre_servicio or not str(nombre_servicio).strip():  # Verifica que exista nombre del servicio
            raise ParametroFaltanteError("El nombre del servicio es obligatorio.")
        try:
            tarifa_num = float(tarifa_base)  # Intenta convertir la tarifa a número flotante
        except (TypeError, ValueError) as exc:
            raise DatosInvalidosError(f"La tarifa base '{tarifa_base}' debe ser numérica.") from exc  # Encadenamiento de excepción
        if tarifa_num <= 0:  # Valida que la tarifa sea estrictamente positiva
            raise DatosInvalidosError(f"La tarifa base debe ser mayor a 0. Recibido: {tarifa_num}")

        self._nombre_servicio = str(nombre_servicio).strip()  # Guarda el nombre del servicio
        self._tarifa_base = tarifa_num  # Guarda la tarifa base validada
        self._disponible = bool(disponible)  # Guarda el estado de disponibilidad

    @property
    def nombre_servicio(self) -> str:
        """Retorna el nombre comercial del servicio."""
        return self._nombre_servicio  # Devuelve nombre

    @property
    def tarifa_base(self) -> float:
        """Retorna la tarifa base por hora/unidad."""
        return self._tarifa_base  # Devuelve tarifa base

    @property
    def disponible(self) -> bool:
        """Indica si el servicio está habilitado para reservas."""
        return self._disponible  # Devuelve estado booleano

    @disponible.setter
    def disponible(self, estado: bool):
        """Permite activar o desactivar la disponibilidad del servicio."""
        self._disponible = bool(estado)  # Actualiza disponibilidad

    @abstractmethod
    def validar_parametros(self, duracion_horas: int) -> None:
        """Valida reglas específicas según el tipo de servicio y duración."""
        pass  # Implementación polimórfica en subclases

    @abstractmethod
    def calcular_costo(self, duracion_horas: int) -> float:
        """Calcula el subtotal base según la duración y naturaleza del servicio."""
        pass  # Implementación polimórfica en subclases

    @abstractmethod
    def describir_servicio(self) -> str:
        """Retorna una descripción técnica detallada del servicio."""
        pass  # Implementación polimórfica en subclases

    def calcular_costo_detallado(
        self,
        duracion_horas: int,
        impuesto_pct: float = 0.0,
        descuento_pct: float = 0.0,
        cargo_extra: float = 0.0
    ) -> float:
        """
        Método sobrecargado mediante parámetros opcionales para calcular el costo final
        aplicando impuestos, descuentos y cargos adicionales con validación de consistencia.
        """
        self.validar_parametros(duracion_horas)  # Valida duración antes de calcular
        if impuesto_pct < 0 or descuento_pct < 0 or descuento_pct > 100 or cargo_extra < 0:  # Valida rangos porcentuales
            raise CalculoInconsistenteError(
                f"Parámetros financieros fuera de rango: IVA={impuesto_pct}%, Desc={descuento_pct}%, Extra={cargo_extra}"
            )
        subtotal = self.calcular_costo(duracion_horas)  # Obtiene costo polimórfico base
        monto_descuento = subtotal * (descuento_pct / 100.0)  # Calcula valor del descuento
        base_gravable = subtotal - monto_descuento + cargo_extra  # Calcula base después de descuento y extras
        total = base_gravable * (1.0 + impuesto_pct / 100.0)  # Aplica impuesto sobre la base gravable
        if total <= 0:  # Comprueba que el resultado final no sea cero ni negativo
            raise CalculoInconsistenteError(f"El costo total calculado resultó inválido: {total}")
        return round(total, 2)  # Retorna valor redondeado a 2 decimales

    def obtener_resumen(self) -> str:
        """Implementa obtener_resumen delegando en describir_servicio."""
        return self.describir_servicio()  # Reutiliza la descripción polimórfica


class ReservaSala(Servicio):
    """Servicio especializado para reserva de salas de reuniones o capacitación."""

    def __init__(self, id_servicio: str, nombre_servicio: str, tarifa_base: float, capacidad: int, disponible: bool = True):
        super().__init__(id_servicio, nombre_servicio, tarifa_base, disponible)  # Inicializa clase padre Servicio
        if not isinstance(capacidad, int) or capacidad <= 0:  # Valida capacidad entera positiva
            raise DatosInvalidosError(f"La capacidad de la sala debe ser un entero positivo. Recibido: {capacidad}")
        self._capacidad = capacidad  # Guarda capacidad máxima de personas
        logger.info(f"Servicio ReservaSala creado: [{self.id}] {self.nombre_servicio} (Capacidad: {self._capacidad})")

    def validar_parametros(self, duracion_horas: int) -> None:
        """Valida que la reserva de sala sea de 1 a 12 horas."""
        if not isinstance(duracion_horas, int) or duracion_horas < 1 or duracion_horas > 12:  # Verifica rango permitido
            raise DatosInvalidosError(f"Duración inválida para sala ({duracion_horas}h). Debe estar entre 1 y 12 horas.")

    def calcular_costo(self, duracion_horas: int) -> float:
        """Calcula costo de sala; aplica 10% de tarifa preferencial si supera 4 horas."""
        self.validar_parametros(duracion_horas)  # Ejecuta validación específica
        costo = self.tarifa_base * duracion_horas  # Calcula tarifa lineal
        if duracion_horas >= 5:  # Aplica beneficio por bloque extendido
            costo *= 0.90  # Descuento automático del 10% por jornada larga
        return round(costo, 2)  # Retorna subtotal

    def describir_servicio(self) -> str:
        """Describe la sala incluyendo capacidad y tarifa."""
        estado = "Disponible" if self.disponible else "No disponible"  # Traduce estado booleano
        return f"Sala [{self.id}] '{self.nombre_servicio}' | Capacidad: {self._capacidad} pers. | ${self.tarifa_base}/h | {estado}"


class AlquilerEquipo(Servicio):
    """Servicio especializado para alquiler de estaciones de trabajo, servidores o proyectores."""

    def __init__(self, id_servicio: str, nombre_servicio: str, tarifa_base: float, categoria_hardware: str, deposito_seguridad: float, disponible: bool = True):
        super().__init__(id_servicio, nombre_servicio, tarifa_base, disponible)  # Inicializa clase padre Servicio
        if not categoria_hardware or not str(categoria_hardware).strip():  # Valida categoría del equipo
            raise ParametroFaltanteError("La categoría de hardware es obligatoria para AlquilerEquipo.")
        if deposito_seguridad < 0:  # Valida depósito no negativo
            raise DatosInvalidosError("El depósito de seguridad no puede ser negativo.")
        self._categoria_hardware = str(categoria_hardware).strip()  # Guarda categoría
        self._deposito_seguridad = float(deposito_seguridad)  # Guarda depósito obligatorio
        logger.info(f"Servicio AlquilerEquipo creado: [{self.id}] {self.nombre_servicio} ({self._categoria_hardware})")

    def validar_parametros(self, duracion_horas: int) -> None:
        """Valida que el alquiler de equipo sea mínimo de 2 horas y máximo de 72 horas."""
        if not isinstance(duracion_horas, int) or duracion_horas < 2 or duracion_horas > 72:  # Verifica límites de alquiler
            raise DatosInvalidosError(f"Duración inválida para alquiler de equipo ({duracion_horas}h). Rango: 2 a 72 horas.")

    def calcular_costo(self, duracion_horas: int) -> float:
        """Calcula el costo del alquiler sumando el depósito reembolsable de seguridad."""
        self.validar_parametros(duracion_horas)  # Ejecuta validación específica
        return round((self.tarifa_base * duracion_horas) + self._deposito_seguridad, 2)  # Suma tarifa por hora + depósito

    def describir_servicio(self) -> str:
        """Describe el equipo tecnológico y su depósito."""
        estado = "Disponible" if self.disponible else "No disponible"  # Traduce estado
        return f"Equipo [{self.id}] '{self.nombre_servicio}' ({self._categoria_hardware}) | ${self.tarifa_base}/h + Depósito ${self._deposito_seguridad} | {estado}"


class AsesoriaEspecializada(Servicio):
    """Servicio especializado de consultoría en arquitectura de software, nube o ciberseguridad."""

    def __init__(self, id_servicio: str, nombre_servicio: str, tarifa_base: float, especialidad: str, nivel_senior: bool = True, disponible: bool = True):
        super().__init__(id_servicio, nombre_servicio, tarifa_base, disponible)  # Inicializa clase padre Servicio
        if not especialidad or not str(especialidad).strip():  # Valida campo especialidad
            raise ParametroFaltanteError("La especialidad es obligatoria para AsesoriaEspecializada.")
        self._especialidad = str(especialidad).strip()  # Guarda área de conocimiento
        self._nivel_senior = bool(nivel_senior)  # Indica si incluye recargo por consultor senior
        logger.info(f"Servicio AsesoriaEspecializada creado: [{self.id}] {self.nombre_servicio} ({self._especialidad})")

    def validar_parametros(self, duracion_horas: int) -> None:
        """Valida que las asesorías sean bloques de 1 a 8 horas."""
        if not isinstance(duracion_horas, int) or duracion_horas < 1 or duracion_horas > 8:  # Verifica duración
            raise DatosInvalidosError(f"Duración inválida para asesoría ({duracion_horas}h). Rango permitido: 1 a 8 horas.")

    def calcular_costo(self, duracion_horas: int) -> float:
        """Calcula costo de asesoría; aplica recargo del 25% si el consultor es nivel Senior."""
        self.validar_parametros(duracion_horas)  # Ejecuta validación específica
        factor = 1.25 if self._nivel_senior else 1.0  # Define multiplicador por experiencia senior
        return round(self.tarifa_base * duracion_horas * factor, 2)  # Retorna valor total de consultoría

    def describir_servicio(self) -> str:
        """Describe la asesoría especializada."""
        nivel = "Senior" if self._nivel_senior else "Semi-Senior"  # Etiqueta nivel del consultor
        estado = "Disponible" if self.disponible else "No disponible"  # Traduce disponibilidad
        return f"Asesoría [{self.id}] '{self.nombre_servicio}' [{self._especialidad} - {nivel}] | ${self.tarifa_base}/h | {estado}"


print("Jerarquía de servicios especializados cargada correctamente.")

Jerarquía de servicios especializados cargada correctamente.

In [4]:
# ==============================================================================
# 6. CLASE RESERVA Y GESTOR INTEGRAL DE SOFTWARE FJ (SIN BASE DE DATOS)
# ==============================================================================
class Reserva(EntidadSistema):
    """Integra Cliente, Servicio y duración, gestionando estados y ciclo de vida."""

    ESTADOS_VALIDOS = ("PENDIENTE", "CONFIRMADA", "PROCESADA", "CANCELADA")  # Tupla de estados permitidos

    def __init__(self, id_reserva: str, cliente: Cliente, servicio: Servicio, duracion_horas: int):
        super().__init__(id_reserva)  # Llama al constructor de EntidadSistema
        if not isinstance(cliente, Cliente):  # Valida instancia de Cliente
            raise DatosInvalidosError("Se requiere una instancia válida de Cliente para crear la reserva.")
        if not isinstance(servicio, Servicio):  # Valida instancia de Servicio
            raise DatosInvalidosError("Se requiere una instancia válida de Servicio para crear la reserva.")
        if not servicio.disponible:  # Verifica disponibilidad inmediata del servicio
            raise ServicioNoDisponibleError(f"El servicio '{servicio.nombre_servicio}' [{servicio.id}] no está disponible.")

        servicio.validar_parametros(duracion_horas)  # Valida duración según el tipo de servicio
        self._cliente = cliente  # Asocia el objeto Cliente
        self._servicio = servicio  # Asocia el objeto Servicio
        self._duracion_horas = duracion_horas  # Guarda duración en horas
        self._estado = "PENDIENTE"  # Estado inicial al instanciar
        self._costo_total = servicio.calcular_costo_detallado(duracion_horas, impuesto_pct=19.0)  # Calcula costo con IVA 19%

    @property
    def estado(self) -> str:
        """Retorna el estado actual de la reserva."""
        return self._estado  # Devuelve estado

    @property
    def costo_total(self) -> float:
        """Retorna el costo total calculado de la reserva."""
        return self._costo_total  # Devuelve monto total

    def confirmar(self) -> None:
        """Cambia el estado de PENDIENTE a CONFIRMADA validando reglas de negocio."""
        if self._estado != "PENDIENTE":  # Solo una reserva pendiente puede confirmarse
            raise ReservaInvalidaError(f"No se puede confirmar la reserva [{self.id}] porque su estado es '{self._estado}'.")
        if not self._servicio.disponible:  # Revalida disponibilidad antes de confirmar
            raise ServicioNoDisponibleError(f"El servicio '{self._servicio.nombre_servicio}' dejó de estar disponible.")
        self._estado = "CONFIRMADA"  # Actualiza estado a CONFIRMADA
        self._cliente.agregar_reserva_historial(self.id)  # Registra en el historial del cliente
        logger.info(f"Reserva CONFIRMADA: [{self.id}] Cliente={self._cliente.nombre} | Servicio={self._servicio.nombre_servicio} | Total=${self._costo_total}")

    def procesar(self, descuento_pct: float = 0.0) -> None:
        """Procesa y cierra una reserva previamente confirmada, aplicando descuento opcional."""
        if self._estado != "CONFIRMADA":  # Exige confirmación previa para procesar
            raise ReservaInvalidaError(f"No se puede procesar la reserva [{self.id}] en estado '{self._estado}'. Debe estar CONFIRMADA.")
        if descuento_pct > 0:  # Recalcula si existe un descuento promocional
            self._costo_total = self._servicio.calcular_costo_detallado(
                self._duracion_horas, impuesto_pct=19.0, descuento_pct=descuento_pct
            )
        self._estado = "PROCESADA"  # Marca la reserva como ejecutada/procesada
        logger.info(f"Reserva PROCESADA: [{self.id}] Valor final cobrado=${self._costo_total}")

    def cancelar(self, motivo: str) -> None:
        """Cancela una reserva si aún no ha sido procesada."""
        if not motivo or not str(motivo).strip():  # Exige justificación de cancelación
            raise ParametroFaltanteError("Debe indicar un motivo válido para cancelar la reserva.")
        if self._estado == "PROCESADA":  # Impide cancelar servicios ya prestados
            raise ReservaInvalidaError(f"La reserva [{self.id}] ya fue PROCESADA y no puede cancelarse.")
        if self._estado == "CANCELADA":  # Evita doble cancelación
            raise ReservaInvalidaError(f"La reserva [{self.id}] ya se encuentra CANCELADA.")
        self._estado = "CANCELADA"  # Cambia estado a CANCELADA
        logger.warning(f"Reserva CANCELADA: [{self.id}] Motivo: {motivo}")

    def obtener_resumen(self) -> str:
        """Retorna una cadena legible con los datos completos de la reserva."""
        return (
            f"Reserva [{self.id}] | Estado: {self._estado} | Cliente: {self._cliente.nombre} | "
            f"Servicio: {self._servicio.nombre_servicio} ({self._duracion_horas}h) | Total: ${self._costo_total}"
        )


class SistemaGestionSoftwareFJ:
    """Controlador principal que administra clientes, servicios y reservas en memoria (listas)."""

    def __init__(self):
        self.clientes: List[Cliente] = []  # Lista interna en memoria para clientes
        self.servicios: List[Servicio] = []  # Lista interna en memoria para servicios
        self.reservas: List[Reserva] = []  # Lista interna en memoria para reservas
        logger.info("=== INICIO DEL SISTEMA INTEGRAL SOFTWARE FJ ===")  # Marca inicio en el archivo de logs

    def registrar_cliente(self, cliente: Cliente) -> None:
        """Agrega un cliente verificando que no exista duplicado por ID."""
        if any(c.id == cliente.id for c in self.clientes):  # Comprueba unicidad de ID en lista
            raise DatosInvalidosError(f"Ya existe un cliente registrado con el ID '{cliente.id}'.")
        self.clientes.append(cliente)  # Añade el objeto a la lista interna

    def registrar_servicio(self, servicio: Servicio) -> None:
        """Agrega un servicio verificando que no exista duplicado por ID."""
        if any(s.id == servicio.id for s in self.servicios):  # Comprueba unicidad de ID de servicio
            raise DatosInvalidosError(f"Ya existe un servicio registrado con el ID '{servicio.id}'.")
        self.servicios.append(servicio)  # Añade el servicio a la lista interna

    def crear_reserva(self, id_reserva: str, id_cliente: str, id_servicio: str, duracion_horas: int) -> Reserva:
        """Busca cliente y servicio en las listas internas y crea una reserva controlada."""
        cliente = next((c for c in self.clientes if c.id == id_cliente), None)  # Busca cliente por ID
        if cliente is None:  # Lanza excepción si el cliente no existe
            raise DatosInvalidosError(f"No se encontró ningún cliente con ID '{id_cliente}'.")

        servicio = next((s for s in self.servicios if s.id == id_servicio), None)  # Busca servicio por ID
        if servicio is None:  # Lanza excepción si el servicio no existe
            raise DatosInvalidosError(f"No se encontró ningún servicio con ID '{id_servicio}'.")

        if any(r.id == id_reserva for r in self.reservas):  # Evita IDs de reserva repetidos
            raise DatosInvalidosError(f"Ya existe una reserva con ID '{id_reserva}'.")

        nueva_reserva = Reserva(id_reserva, cliente, servicio, duracion_horas)  # Instancia la reserva
        self.reservas.append(nueva_reserva)  # Almacena en la lista interna de reservas
        logger.info(f"Reserva creada en estado PENDIENTE: [{nueva_reserva.id}]")  # Documenta en log
        return nueva_reserva  # Devuelve el objeto reserva


print("Clases Reserva y SistemaGestionSoftwareFJ cargadas correctamente.")

Clases Reserva y SistemaGestionSoftwareFJ cargadas correctamente.

In [5]:
# ==============================================================================
# 7. SIMULACIÓN DE 12 OPERACIONES COMPLETAS (VÁLIDAS E INVÁLIDAS)
# Demostración de try/except, try/except/else, try/except/finally y encadenamiento
# ==============================================================================
sistema = SistemaGestionSoftwareFJ()  # Instancia el sistema principal en memoria

def ejecutar_operacion_segura(numero_op: int, descripcion: str, funcion_op):
    """Ejecuta cada operación dentro de bloques try/except/else/finally sin detener el programa."""
    print(f"--- Operación #{numero_op:02d}: {descripcion} ---")  # Imprime encabezado de la prueba
    try:
        resultado = funcion_op()  # Ejecuta la función de prueba pasada como argumento
    except ErrorSoftwareFJ as err:
        # Captura excepciones propias del dominio de negocio y las registra en el log
        mensaje_log = f"[OP-{numero_op:02d}] {err.codigo_error} ({type(err).__name__}): {err}"
        logger.error(mensaje_log)  # Escribe el error controlado en software_fj.log
        print(f"  [ERROR CONTROLADO] {mensaje_log}")  # Muestra retroalimentación en consola
    except Exception as exc:
        # Captura cualquier error inesperado y lo encadena a ErrorSoftwareFJ
        try:
            raise ErrorSoftwareFJ(f"Fallo inesperado en operación {numero_op}: {exc}", "ERR-999") from exc
        except ErrorSoftwareFJ as encadenado:
            logger.critical(f"[OP-{numero_op:02d}] {encadenado.codigo_error}: {encadenado} | Causa: {encadenado.__cause__}")
            print(f"  [ERROR CRÍTICO ENCADENADO] {encadenado}")
    else:
        # Bloque else: se ejecuta únicamente cuando no ocurrió ninguna excepción
        print(f"  [ÉXITO] {resultado}")  # Confirma ejecución exitosa en consola
    finally:
        # Bloque finally: se ejecuta siempre para auditar el cierre de la operación
        logger.debug(f"[OP-{numero_op:02d}] Ciclo de ejecución finalizado (bloque finally).")


# OP 01: Registro válido de cliente 1
ejecutar_operacion_segura(
    1, "Registro válido de cliente (Laura Martínez)",
    lambda: (sistema.registrar_cliente(Cliente("CLI-01", "Laura Martínez", "laura.martinez@empresa.co", "3104567890")), "Cliente CLI-01 registrado.")[1]
)

# OP 02: Registro válido de cliente 2
ejecutar_operacion_segura(
    2, "Registro válido de cliente (Carlos Gómez)",
    lambda: (sistema.registrar_cliente(Cliente("CLI-02", "Carlos Gómez", "cgomez@techbogota.com", "3209876543")), "Cliente CLI-02 registrado.")[1]
)

# OP 03: Registro inválido de cliente (correo mal formado)
ejecutar_operacion_segura(
    3, "Intento de registro de cliente con correo inválido",
    lambda: sistema.registrar_cliente(Cliente("CLI-03", "Andrés Ruiz", "correo_sin_arroba.com", "3001112233"))
)

# OP 04: Registro inválido de cliente (parámetro faltante en nombre)
ejecutar_operacion_segura(
    4, "Intento de registro de cliente con nombre vacío",
    lambda: sistema.registrar_cliente(Cliente("CLI-04", "   ", "valido@dominio.com", "3112223344"))
)

# OP 05: Creación válida de servicios especializados (Sala, Equipo, Asesoría)
def crear_catalogo_servicios():
    s1 = ReservaSala("SRV-01", "Sala Innovación Cloud", 80000.0, capacidad=15, disponible=True)  # Crea sala disponible
    s2 = AlquilerEquipo("SRV-02", "Workstation GPU RTX", 45000.0, categoria_hardware="Cómputo Alto Rendimiento", deposito_seguridad=100000.0, disponible=True)  # Crea equipo disponible
    s3 = AsesoriaEspecializada("SRV-03", "Arquitectura Microservicios", 150000.0, especialidad="Cloud & DevOps", nivel_senior=True, disponible=True)  # Crea asesoría disponible
    s4 = ReservaSala("SRV-04", "Auditorio Principal", 200000.0, capacidad=60, disponible=False)  # Crea sala en mantenimiento (no disponible)
    for srv in (s1, s2, s3, s4):  # Itera sobre los servicios creados
        sistema.registrar_servicio(srv)  # Registra cada servicio en el sistema
    return "4 servicios especializados registrados en catálogo."

ejecutar_operacion_segura(5, "Creación válida de catálogo de servicios especializados", crear_catalogo_servicios)

# OP 06: Creación incorrecta de servicio (tarifa no numérica con encadenamiento)
ejecutar_operacion_segura(
    6, "Intento de creación de servicio con tarifa no numérica",
    lambda: sistema.registrar_servicio(AlquilerEquipo("SRV-ERR", "Servidor Rack", "cien_mil", "Redes", 50000.0))
)

# OP 07: Reserva exitosa de Sala (6 horas, aplica descuento por jornada larga + IVA) y confirmación
def flujo_reserva_sala_exitosa():
    res = sistema.crear_reserva("RES-01", "CLI-01", "SRV-01", duracion_horas=6)  # Crea reserva válida de 6 horas
    res.confirmar()  # Confirma la reserva
    res.procesar(descuento_pct=5.0)  # Procesa con 5% de descuento adicional
    return res.obtener_resumen()  # Retorna resumen final

ejecutar_operacion_segura(7, "Reserva, confirmación y procesamiento exitoso de Sala", flujo_reserva_sala_exitosa)

# OP 08: Reserva exitosa de Alquiler de Equipo (10 horas) y cancelación controlada
def flujo_reserva_equipo_cancelada():
    res = sistema.crear_reserva("RES-02", "CLI-02", "SRV-02", duracion_horas=10)  # Crea reserva válida de equipo
    res.confirmar()  # Confirma reserva
    res.cancelar("Cambio de cronograma del cliente")  # Cancela con motivo válido
    return res.obtener_resumen()  # Retorna resumen

ejecutar_operacion_segura(8, "Reserva de Equipo confirmada y posteriormente cancelada", flujo_reserva_equipo_cancelada)

# OP 09: Intento fallido de reservar un servicio NO DISPONIBLE (SRV-04)
ejecutar_operacion_segura(
    9, "Intento de reserva sobre servicio no disponible (Auditorio)",
    lambda: sistema.crear_reserva("RES-03", "CLI-01", "SRV-04", duracion_horas=3)
)

# OP 10: Intento fallido de reservar con duración fuera de rango (Asesoría de 15 horas, máximo 8h)
ejecutar_operacion_segura(
    10, "Intento de reserva de Asesoría con duración inválida (15h)",
    lambda: sistema.crear_reserva("RES-04", "CLI-02", "SRV-03", duracion_horas=15)
)

# OP 11: Intento fallido de procesar una reserva que ya fue CANCELADA (RES-02)
ejecutar_operacion_segura(
    11, "Intento de procesar reserva previamente cancelada (RES-02)",
    lambda: sistema.reservas[1].procesar()
)

# OP 12: Cálculo inconsistente (descuento mayor al 100%)
ejecutar_operacion_segura(
    12, "Validación de cálculo inconsistente con descuento del 150%",
    lambda: sistema.servicios[2].calcular_costo_detallado(duracion_horas=2, impuesto_pct=19.0, descuento_pct=150.0)
)

print("\n==============================================================================")
print("RESUMEN FINAL DE ESTADO EN MEMORIA (SIN BASE DE DATOS)")
print("==============================================================================")
print(f"Total Clientes activos  : {len(sistema.clientes)}")
print(f"Total Servicios catálogo: {len(sistema.servicios)}")
print(f"Total Reservas creadas  : {len(sistema.reservas)}")
for r in sistema.reservas:
    print(" ->", r.obtener_resumen())

--- Operación #01: Registro válido de cliente (Laura Martínez) ---
  [ÉXITO] Cliente CLI-01 registrado.
--- Operación #02: Registro válido de cliente (Carlos Gómez) ---
  [ÉXITO] Cliente CLI-02 registrado.
--- Operación #03: Intento de registro de cliente con correo inválido ---
  [ERROR CONTROLADO] [OP-03] ERR-101 (DatosInvalidosError): Formato de correo electrónico inválido: 'correo_sin_arroba.com'.
--- Operación #04: Intento de registro de cliente con nombre vacío ---
  [ERROR CONTROLADO] [OP-04] ERR-102 (ParametroFaltanteError): El nombre del cliente es obligatorio.
--- Operación #05: Creación válida de catálogo de servicios especializados ---
  [ÉXITO] 4 servicios especializados registrados en catálogo.
--- Operación #06: Intento de creación de servicio con tarifa no numérica ---
  [ERROR CONTROLADO] [OP-06] ERR-101 (DatosInvalidosError): La tarifa base 'cien_mil' debe ser numérica.
--- Operación #07: Reserva, confirmación y procesamiento exitoso de Sala ---
  [ÉXITO] Reserva [RES

In [6]:
# ==============================================================================
# 8. VERIFICACIÓN DEL ARCHIVO DE LOGS GENERADO (software_fj.log)
# ==============================================================================
print("Contenido registrado en el archivo de logs 'software_fj.log':\n")
with open(LOG_FILENAME, "r", encoding="utf-8") as f:  # Abre el archivo de logs en modo lectura
    print(f.read())  # Imprime todas las líneas de auditoría y errores capturados

Contenido registrado en el archivo de logs 'software_fj.log':

2026-11-21 17:19:08 | INFO     | === INICIO DEL SISTEMA INTEGRAL SOFTWARE FJ ===
2026-11-21 17:19:08 | INFO     | Cliente registrado exitosamente: [CLI-01] Laura Martínez
2026-11-21 17:19:08 | DEBUG    | [OP-01] Ciclo de ejecución finalizado (bloque finally).
2026-11-21 17:19:08 | INFO     | Cliente registrado exitosamente: [CLI-02] Carlos Gómez
2026-11-21 17:19:08 | DEBUG    | [OP-02] Ciclo de ejecución finalizado (bloque finally).
2026-11-21 17:19:08 | ERROR    | [OP-03] ERR-101 (DatosInvalidosError): Formato de correo electrónico inválido: 'correo_sin_arroba.com'.
2026-11-21 17:19:08 | DEBUG    | [OP-03] Ciclo de ejecución finalizado (bloque finally).
2026-11-21 17:19:08 | ERROR    | [OP-04] ERR-102 (ParametroFaltanteError): El nombre del cliente es obligatorio.
2026-11-21 17:19:08 | DEBUG    | [OP-04] Ciclo de ejecución finalizado (bloque finally).
2026-11-21 17:19:08 | INFO     | Servicio ReservaSala creado: [SRV-01] S